In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

In [2]:
df = pd.read_csv('first_25000_rows.csv')
len(df)

5000

In [3]:
df['ts_event'].nunique()

4810

We have multiple non-unique timestamps here, so I decide to use the last row of each timestamp whenever we encounter those:

In [4]:
df = df.sort_values('ts_event').groupby('ts_event').last().reset_index()

In [5]:
def compute_ofi(row, levels=[0]):
    ofi = 0
    for level in levels:
        # Skip first timestamp
        if pd.isna(row[f'prev_bid_px_0{level}']) or pd.isna(row[f'prev_ask_px_0{level}']):
            continue

        if row[f'bid_px_0{level}'] > row[f'prev_bid_px_0{level}']:
            ofi_bid = row[f'bid_sz_0{level}']
        elif row[f'bid_px_0{level}'] == row[f'prev_bid_px_0{level}']:
            ofi_bid = row[f'bid_sz_0{level}'] - row[f'prev_bid_sz_0{level}']
        else:
            ofi_bid = -row[f'bid_sz_0{level}']

        if row[f'ask_px_0{level}'] > row[f'prev_ask_px_0{level}']:
            ofi_ask = -row[f'ask_sz_0{level}']
        elif row[f'ask_px_0{level}'] == row[f'prev_ask_px_0{level}']:
            ofi_ask = row[f'ask_sz_0{level}'] - row[f'prev_ask_sz_0{level}']
        else:
            ofi_ask = row[f'ask_sz_0{level}']

        ofi += ofi_bid - ofi_ask
    return ofi

## Best-level OFI

In [6]:
best_lvl = df[['ts_event','bid_px_00', 'ask_px_00', 'bid_sz_00', 'ask_sz_00']].copy()
best_lvl['prev_bid_px_00'] = best_lvl['bid_px_00'].shift(1)
best_lvl['prev_ask_px_00'] = best_lvl['ask_px_00'].shift(1)
best_lvl['prev_bid_sz_00'] = best_lvl['bid_sz_00'].shift(1)
best_lvl['prev_ask_sz_00'] = best_lvl['ask_sz_00'].shift(1)

best_lvl['ts_event'] = pd.to_datetime(best_lvl['ts_event'])
best_lvl.set_index('ts_event', inplace=True)
best_lvl

,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,prev_bid_px_00,prev_ask_px_00,prev_bid_sz_00,prev_ask_sz_00
ts_event,,,,,,,,
2024-10-21 11:54:29.221064336+00:00,233.67,233.74,139,200,NaN,NaN,NaN,NaN
2024-10-21 11:54:29.223769812+00:00,233.67,233.74,141,200,233.67,233.74,139.0,200.0
2024-10-21 11:54:29.225030400+00:00,233.67,233.74,144,200,233.67,233.74,141.0,200.0
2024-10-21 11:54:29.712434212+00:00,233.67,233.74,144,200,233.67,233.74,144.0,200.0
2024-10-21 11:54:29.764673165+00:00,233.67,233.74,144,200,233.67,233.74,144.0,200.0
...,...,...,...,...,...,...,...,...
2024-10-21 13:04:16.583527688+00:00,233.51,233.61,1,20,233.51,233.61,1.0,20.0
2024-10-21 13:04:17.976461017+00:00,233.51,233.61,1,20,233.51,233.61,1.0,20.0
2024-10-21 13:04:20.085638629+00:00,233.51,233.61,1,20,233.51,233.61,1.0,20.0


In [7]:
best_lvl['best_lvl_OFI'] = best_lvl.apply(lambda row: compute_ofi(row, levels=[0]), axis=1)
best_lvl['best_lvl_OFI'] = best_lvl['best_lvl_OFI'].rolling('60s').sum()
best_lvl['best_lvl_OFI']

ts_event
2024-10-21 11:54:29.221064336+00:00     0.0
2024-10-21 11:54:29.223769812+00:00     2.0
2024-10-21 11:54:29.225030400+00:00     5.0
2024-10-21 11:54:29.712434212+00:00     5.0
2024-10-21 11:54:29.764673165+00:00     5.0
                                       ... 
2024-10-21 13:04:16.583527688+00:00   -81.0
2024-10-21 13:04:17.976461017+00:00   -81.0
2024-10-21 13:04:20.085638629+00:00   -81.0
2024-10-21 13:04:20.085651109+00:00   -81.0
2024-10-21 13:04:20.130842270+00:00   -95.0
Name: best_lvl_OFI, Length: 4810, dtype: float64

## Multi-level OFI

In [8]:
bid_px_cols = [f"bid_px_0{i}" for i in range(10)]
ask_px_cols = [f"ask_px_0{i}" for i in range(10)]
bid_sz_cols = [f"bid_sz_0{i}" for i in range(10)]
ask_sz_cols = [f"ask_sz_0{i}" for i in range(10)]

relevant_cols = ['ts_event'] + bid_px_cols + ask_px_cols + bid_sz_cols + ask_sz_cols
multi_lvl = df[relevant_cols].copy()

multi_lvl['ts_event'] = pd.to_datetime(multi_lvl['ts_event'])
multi_lvl.set_index('ts_event', inplace=True)

In [9]:
for i in range(10):
    multi_lvl[f'prev_bid_px_0{i}'] = multi_lvl[f'bid_px_0{i}'].shift(1)
    multi_lvl[f'prev_bid_sz_0{i}'] = multi_lvl[f'bid_sz_0{i}'].shift(1)
    multi_lvl[f'prev_ask_px_0{i}'] = multi_lvl[f'ask_px_0{i}'].shift(1)
    multi_lvl[f'prev_ask_sz_0{i}'] = multi_lvl[f'ask_sz_0{i}'].shift(1)

In [10]:
levels = range(10)

for m in levels:
    col_name = f'OFI_lvl_{m}'
    multi_lvl[col_name] = multi_lvl.apply(lambda row: compute_ofi(row, levels=[m]), axis=1)
    multi_lvl[f'{col_name}_h'] = multi_lvl[col_name].rolling('60s').sum()

multi_lvl['depth_sum'] = sum(multi_lvl[f'bid_sz_0{m}'] + multi_lvl[f'ask_sz_0{m}'] for m in levels)

multi_lvl['Q'] = (multi_lvl['depth_sum'].rolling('60s').mean() / 10) / 2 # use mean() to account for delta_N

for m in levels:
    ofi_col = f'OFI_lvl_{m}_h'
    norm_col = f'norm_OFI_lvl_{m}'
    multi_lvl[norm_col] = multi_lvl[ofi_col] / multi_lvl['Q']

norm_ofi_cols = [f'norm_OFI_lvl_{m}' for m in levels]
multi_lvl['ofi_vector'] = multi_lvl[norm_ofi_cols].values.tolist()
multi_lvl['ofi_vector']

ts_event
2024-10-21 11:54:29.221064336+00:00    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
2024-10-21 11:54:29.223769812+00:00    [0.020100502512562814, 0.0, 0.0, 0.0, 0.0, 0.0...
2024-10-21 11:54:29.225030400+00:00    [0.050217609641781055, 0.0, 0.0, 0.0, 0.0, 0.0...
2024-10-21 11:54:29.712434212+00:00    [0.04897159647404506, 0.0, 1.9588638589618024,...
2024-10-21 11:54:29.764673165+00:00    [0.04920291281243849, 0.0, 0.0, 0.0, 0.0, 0.0,...
                                                             ...                        
2024-10-21 13:04:16.583527688+00:00    [-0.7025547445255474, -13.313846084527349, 9.0...
2024-10-21 13:04:17.976461017+00:00    [-0.7015595764880935, -15.027232903788176, 9.0...
2024-10-21 13:04:20.085638629+00:00    [-0.7013563501849569, -15.022879846554323, 8.9...
2024-10-21 13:04:20.085651109+00:00    [-0.700404245660304, -15.002486002723796, 8.91...
2024-10-21 13:04:20.130842270+00:00    [-0.8205899289781816, -15.159319214281144, 5.4...
Name: ofi_ve

## Integrated OFI

In [11]:
ofi_matrix = np.array(multi_lvl['ofi_vector'].copy().tolist())
integrated_df = pd.DataFrame(df['ts_event'].copy())
integrated_df.set_index('ts_event', inplace=True)

In [12]:
pca = PCA(n_components=1)
_ = pca.fit_transform(ofi_matrix)

w = pca.components_[0]
w_1 = w / np.sum(np.abs(w))

integrated_df['integrated_ofi'] = ofi_matrix @ w_1
integrated_df

,integrated_ofi
ts_event,
2024-10-21T11:54:29.221064336Z,0.000000
2024-10-21T11:54:29.223769812Z,0.000127
2024-10-21T11:54:29.225030400Z,0.000317
2024-10-21T11:54:29.712434212Z,-0.420971
2024-10-21T11:54:29.764673165Z,0.000311
...,...
2024-10-21T13:04:16.583527688Z,0.803609
2024-10-21T13:04:17.976461017Z,0.724233
2024-10-21T13:04:20.085638629Z,0.496625


## Cross-Asset OFI
I don't think this can be computed since we only have one asset in the dataset